## cấu trúc
```
PromotionSyncService
        ↓
ISyncStrategy
   ↓        ↓
RK7DirectStrategy
RefAgentStrategy
```

## flow
```
promotion
   ↓
detect rule variant
   ↓
resolve strategy
   ↓
strategy.apply()
   ↓
final discount
```

## cấu trúc thư mục
```
services
└─ promotion-sync
   │
   ├─ promotion-sync-service.cs
   │
   ├─ models
   │  └─ promotion-sync-context.cs
   │
   └─ strategies
      │
      ├─ isync-strategy.cs
      ├─ sync-strategy-resolver.cs
      │
      ├─ rk7
      │  └─ rk7-direct-strategy.cs
      │
      └─ refagent
         └─ refagent-strategy.cs
```

In [ ]:

#r "nuget: Microsoft.Extensions.DependencyInjection, 8.0.0",
using System;
using System.Collections.Generic;
using System.Threading.Tasks;
using Microsoft.Extensions.DependencyInjection;


// =========================================================
// DOMAIN
// =========================================================

public class PromotionSyncContext
{
    public string PromotionName { get; set; }
    public string RK7Payload { get; set; }
    public string Route { get; set; }
}


// =========================================================
// STRATEGY CONTRACT
// =========================================================

public interface ISyncStrategy
{
    Task Execute(PromotionSyncContext context);
}


// =========================================================
// SERVICES
// =========================================================

public interface IRk7Service
{
    Task Send(string payload);
}

public class Rk7Service : IRk7Service
{
    public Task Send(string payload)
    {
        Console.WriteLine("calling rk7 api...");
        Console.WriteLine(payload);
        return Task.CompletedTask;
    }
}

public interface IRefAgentService
{
    Task Send(string payload);
}

public class RefAgentService : IRefAgentService
{
    public async Task Send(string payload)
    {
        Console.WriteLine("calling refagent api...");
        await Task.Delay(10);

        Console.WriteLine("refagent → rk7");
    }
}


// =========================================================
// STRATEGIES
// =========================================================

public class RK7DirectStrategy : ISyncStrategy
{
    private readonly IRk7Service _rk7;

    public RK7DirectStrategy(IRk7Service rk7)
    {
        _rk7 = rk7;
    }

    public async Task Execute(PromotionSyncContext context)
    {
        Console.WriteLine("sync direct → rk7");

        await _rk7.Send(context.RK7Payload);
    }
}

public class RefAgentStrategy : ISyncStrategy
{
    private readonly IRefAgentService _refAgent;

    public RefAgentStrategy(IRefAgentService refAgent)
    {
        _refAgent = refAgent;
    }

    public async Task Execute(PromotionSyncContext context)
    {
        Console.WriteLine("sync via refagent");

        await _refAgent.Send(context.RK7Payload);
    }
}


// =========================================================
// RESOLVER
// =========================================================

public class SyncStrategyResolver
{
    private readonly IServiceProvider _provider;

    private static readonly Dictionary<string, Type> strategies =
        new()
        {
            { "RK7_DIRECT", typeof(RK7DirectStrategy) },
            { "REFAGENT", typeof(RefAgentStrategy) }
        };

    public SyncStrategyResolver(IServiceProvider provider)
    {
        _provider = provider;
    }

    public ISyncStrategy Resolve(string route)
    {
        if (!strategies.TryGetValue(route, out var type))
            throw new Exception("route not supported");

        return (ISyncStrategy)_provider.GetRequiredService(type);
    }
}


// =========================================================
// SYNC SERVICE
// =========================================================

public class PromotionSyncService
{
    private readonly SyncStrategyResolver _resolver;

    public PromotionSyncService(SyncStrategyResolver resolver)
    {
        _resolver = resolver;
    }

    public async Task Sync(PromotionSyncContext context)
    {
        var strategy = _resolver.Resolve(context.Route);

        await strategy.Execute(context);
    }
}


// =========================================================
// DEMO (MAIN)
// =========================================================

var services = new ServiceCollection();

services.AddTransient<IRk7Service, Rk7Service>();
services.AddTransient<IRefAgentService, RefAgentService>();

services.AddTransient<RK7DirectStrategy>();
services.AddTransient<RefAgentStrategy>();

services.AddSingleton<SyncStrategyResolver>();
services.AddTransient<PromotionSyncService>();

var provider = services.BuildServiceProvider();

var syncService = provider.GetRequiredService<PromotionSyncService>();

var context = new PromotionSyncContext
{
    PromotionName = "summer promo",
    RK7Payload = "<promotion>...</promotion>",
    Route = "RK7_DIRECT"
};

await syncService.Sync(context);